# Tutorial on How to Use FIM-PP

This notebook shows the recommended Hugging Face workflow for the point-process model from {cite}`fim_pp`: load the pretrained model with `AutoModel.from_pretrained(...)`, download a small example dataset from the Hub, prepare the context/inference tensors, visualize the inferred intensities, and finish with a fine-tuning command template.


## Installation

Install the OpenFIM package first so the remote code can import the `fim` modules:

```bash
pip install -e .
```

The tutorial also uses `huggingface_hub` to download the example dataset.


In [ ]:
from pathlib import Path
import warnings

import torch
from huggingface_hub import snapshot_download
from transformers import AutoModel

from fim.trainers.utils import get_accel_type
from point_process_tutorial_helper import load_hawkes_tensors, move_to_device, plot_intensity_comparison, prepare_hawkes_batch

warnings.filterwarnings("ignore")
device = get_accel_type()
device


## Load the Pretrained Model

The standardized user-facing path is now the Transformers AutoModel interface.


In [ ]:
model = AutoModel.from_pretrained("FIM4Science/FIM-PP", trust_remote_code=True)
model = model.to(device)
model.eval()


## Download Example Data

The tutorial dataset is stored as raw tensors on Hugging Face. We download the snapshot and load the `.pt` files directly.


In [ ]:
dataset_root = Path(snapshot_download(repo_id="FIM4Science/10D-Hawkes", repo_type="dataset"))
dataset_root


In [ ]:
tensors = load_hawkes_tensors(dataset_root)
sorted(tensors)


## Build a Context / Inference Batch

We hold out a single path for inference and use the remaining paths as context. The helper also builds a dense evaluation grid for plotting the intensity curves between events.


In [ ]:
batch = prepare_hawkes_batch(tensors, sample_idx=0, inference_path_idx=0, num_points_between_events=10)
batch = move_to_device(batch, device)

for key, value in batch.items():
    if torch.is_tensor(value):
        print(f"{key}: {tuple(value.shape)}")
    else:
        print(f"{key}: {value}")


## Run Zero-Shot Inference


In [ ]:
with torch.no_grad():
    output = model(batch)

sorted(output.keys())


In [ ]:
fig = plot_intensity_comparison(output, batch, path_idx=0)
fig


## Fine-Tuning Starting from FIM-PP

A short fine-tuning run can be started with the existing Hawkes entrypoint. The point-process checkpoint is the initialization source, while the dataset can be either a local tensor folder or an EasyTPP dataset id.

```bash
python scripts/hawkes/fim_finetune.py \
  --config configs/train/hawkes/david.yaml \
  --dataset easytpp/retweet \
  --resume_model /path/to/FIM-PP/model-checkpoint.pth \
  --epochs 200 \
  --val-every 10
```

For local debugging, the lower-level fallback `fim.models.hawkes.FIMHawkes.load_model(...)` is still available, but the primary public workflow should use `AutoModel.from_pretrained(...)`.
